# Implementação de TCN usando Darts

Este notebook implementa uma Temporal Convolutional Network (TCN) para previsão de intercâmbio de energia entre subsistemas elétricos brasileiros, utilizando o pacote Darts.

## Contexto

- **Dados**: Intercâmbio horário entre subsistemas elétricos (2022-2025)
- **Objetivo**: Prever o intercâmbio futuro com base em padrões históricos
- **Modelo**: TCN (Temporal Convolutional Network) via Darts

## Arquitetura TCN

A TCN combina:
1. **Convoluções Causais**: Garantem que não há vazamento de informações futuras
2. **Convoluções Dilatadas**: Expandem o campo receptivo exponencialmente
3. **Blocos Residuais**: Estabilizam o treinamento de redes profundas

In [ ]:
# Imports necessários
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

# Darts imports
from darts import TimeSeries
from darts.models import TCNModel
from darts.dataprocessing.transformers import Scaler
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.metrics import mape, mae, rmse, mse
from pytorch_lightning.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (15, 6)

print("✓ Bibliotecas importadas com sucesso")

## 1. Funções Auxiliares

In [ ]:
def calculate_receptive_field(kernel_size, num_layers, dilation_base=2):
    """
    Calcula o campo receptivo da TCN.
    
    O campo receptivo determina quantos passos de tempo no passado
    o modelo consegue "ver" e utilizar para fazer previsões.
    
    Fórmula: RF = 1 + 2 * (kernel_size - 1) * sum(dilation_base^i for i in range(num_layers))
    """
    dilation_sum = sum(dilation_base ** i for i in range(num_layers))
    rf = 1 + 2 * (kernel_size - 1) * dilation_sum
    return rf

def print_model_config(input_chunk, output_chunk, kernel, layers, filters, dilation=2):
    """Imprime configuração do modelo TCN"""
    rf = calculate_receptive_field(kernel, layers, dilation)
    
    print("=" * 60)
    print("Configuração do Modelo TCN")
    print("=" * 60)
    print(f"Input Chunk Length:      {input_chunk} (horas que o modelo vê)")
    print(f"Output Chunk Length:     {output_chunk} (horas que o modelo prevê)")
    print(f"Kernel Size:             {kernel}")
    print(f"Number of Layers:        {layers}")
    print(f"Number of Filters:       {filters}")
    print(f"Dilation Base:           {dilation}")
    print(f"Receptive Field:         {rf} (horas de memória máxima)")
    print("")
    if input_chunk < rf:
        print(f"⚠️  AVISO: input_chunk_length ({input_chunk}) < RF ({rf})")
        print(f"   O modelo não pode utilizar seu campo receptivo completo!")
    else:
        print(f"✓  input_chunk_length >= RF: Modelo pode usar histórico completo")
    print("=" * 60)

# Teste da função
print_model_config(input_chunk=168, output_chunk=24, kernel=3, layers=4, filters=64)

## 2. Carregamento e Preparação dos Dados

Vamos carregar os dados de intercâmbio de energia e convertê-los para o formato TimeSeries do Darts.

In [ ]:
# Caminho para os dados
# Ajuste este caminho conforme a localização do seu arquivo
data_path = "../data/INTERCAMBIO_NACIONAL_2022-2025.csv"

# Verificar se o arquivo existe
if not Path(data_path).exists():
    print(f"❌ Arquivo não encontrado: {data_path}")
    print("\nPor favor, execute o notebook ONS-data.ipynb primeiro para baixar os dados.")
else:
    print(f"✓ Arquivo encontrado: {data_path}")
    
    # Carregar dados
    df = pd.read_csv(data_path, sep=';')
    
    # Converter timestamp
    df['din_instante'] = pd.to_datetime(df['din_instante'])
    
    print(f"\nDados carregados: {len(df)} registros")
    print(f"Período: {df['din_instante'].min()} até {df['din_instante'].max()}")
    print(f"\nSubsistemas únicos (origem): {df['id_subsistema_origem'].unique()}")
    print(f"Subsistemas únicos (destino): {df['id_subsistema_destino'].unique()}")
    
    # Visualizar primeiras linhas
    display(df.head())
    
    # Estatísticas básicas
    print("\nEstatísticas do intercâmbio (MWmed):")
    display(df['val_intercambiomwmed'].describe())

### 2.1. Seleção e Preparação de uma Série Temporal Específica

Para este exemplo, vamos selecionar um par específico de subsistemas. Você pode modificar isso conforme necessário.

In [ ]:
# Selecionar um par específico de subsistemas
# Exemplo: Nordeste -> Sudeste
ORIGIN = 'NE'
DESTINATION = 'SE'

# Filtrar dados
df_filtered = df[
    (df['id_subsistema_origem'] == ORIGIN) & 
    (df['id_subsistema_destino'] == DESTINATION)
].copy()

# Ordenar por tempo e remover duplicatas
df_filtered = df_filtered.sort_values('din_instante')
df_filtered = df_filtered.drop_duplicates(subset=['din_instante'], keep='first')

print(f"Série selecionada: {ORIGIN} -> {DESTINATION}")
print(f"Registros: {len(df_filtered)}")
print(f"Período: {df_filtered['din_instante'].min()} até {df_filtered['din_instante'].max()}")

# Verificar valores ausentes
print(f"\nValores nulos: {df_filtered['val_intercambiomwmed'].isnull().sum()}")

# Plotar série temporal
plt.figure(figsize=(15, 6))
plt.plot(df_filtered['din_instante'], df_filtered['val_intercambiomwmed'], linewidth=0.5)
plt.title(f'Intercâmbio de Energia: {ORIGIN} → {DESTINATION}', fontsize=14)
plt.xlabel('Data', fontsize=12)
plt.ylabel('Intercâmbio (MWmed)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.2. Converter para TimeSeries do Darts

O Darts trabalha com objetos `TimeSeries`, que garantem consistência temporal e facilitam operações.

In [ ]:
# Criar TimeSeries
ts = TimeSeries.from_dataframe(
    df_filtered,
    time_col='din_instante',
    value_cols='val_intercambiomwmed',
    freq='H'  # Frequência horária
)

print(f"TimeSeries criada:")
print(f"  - Comprimento: {len(ts)} pontos")
print(f"  - Início: {ts.start_time()}")
print(f"  - Fim: {ts.end_time()}")
print(f"  - Frequência: {ts.freq}")

# Plotar usando método do Darts
ts.plot()
plt.title(f'TimeSeries: {ORIGIN} → {DESTINATION}', fontsize=14)
plt.ylabel('Intercâmbio (MWmed)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

## 3. Divisão Cronológica dos Dados

**Importante**: Em séries temporais, a divisão deve respeitar a ordem temporal para evitar vazamento de dados.

Divisão sugerida:
- **Treinamento**: 70% dos dados
- **Validação**: 15% dos dados  
- **Teste**: 15% dos dados

In [ ]:
# Calcular pontos de divisão
n = len(ts)
train_size = int(0.7 * n)
val_size = int(0.15 * n)

# Dividir usando índices temporais
train = ts[:train_size]
val = ts[train_size:train_size + val_size]
test = ts[train_size + val_size:]

print("Divisão dos dados:")
print(f"\nTreinamento:")
print(f"  - Pontos: {len(train)} ({len(train)/n*100:.1f}%)")
print(f"  - Período: {train.start_time()} até {train.end_time()}")

print(f"\nValidação:")
print(f"  - Pontos: {len(val)} ({len(val)/n*100:.1f}%)")
print(f"  - Período: {val.start_time()} até {val.end_time()}")

print(f"\nTeste:")
print(f"  - Pontos: {len(test)} ({len(test)/n*100:.1f}%)")
print(f"  - Período: {test.start_time()} até {test.end_time()}")

# Visualizar divisão
plt.figure(figsize=(15, 6))
train.plot(label='Treinamento')
val.plot(label='Validação')
test.plot(label='Teste')
plt.title('Divisão Temporal dos Dados', fontsize=14)
plt.ylabel('Intercâmbio (MWmed)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## 4. Criação de Covariáveis Temporais

As covariáveis temporais (hora, dia da semana, mês) ajudam o modelo a capturar padrões sazonais.

**Nota**: TCN do Darts suporta apenas `past_covariates`, não `future_covariates`.

In [ ]:
# Criar covariáveis temporais (one-hot encoded)
# Estas são determinísticas e conhecidas para todo o futuro

# Hora do dia (0-23) - 24 features
hour_cov = datetime_attribute_timeseries(ts, attribute='hour', one_hot=True)
print(f"Covariável 'hora': {hour_cov.width} features (one-hot de 24 horas)")

# Dia da semana (0-6) - 7 features
dow_cov = datetime_attribute_timeseries(ts, attribute='dayofweek', one_hot=True)
print(f"Covariável 'dia da semana': {dow_cov.width} features (one-hot de 7 dias)")

# Mês (1-12) - 12 features
month_cov = datetime_attribute_timeseries(ts, attribute='month', one_hot=True)
print(f"Covariável 'mês': {month_cov.width} features (one-hot de 12 meses)")

# Combinar todas as covariáveis
covariates = hour_cov.stack(dow_cov).stack(month_cov)
print(f"\nCovariáveis totais: {covariates.width} features ({24 + 7 + 12})")
print(f"Comprimento: {len(covariates)} (mesmo que a série principal)")

## 5. Normalização dos Dados

**Crucial**: A normalização deve ser feita APENAS com base nos dados de treinamento para evitar vazamento de informações.

In [ ]:
# Criar scaler e ajustar APENAS nos dados de treinamento
scaler = Scaler()
train_scaled = scaler.fit_transform(train)
val_scaled = scaler.transform(val)
test_scaled = scaler.transform(test)

print("✓ Dados normalizados")
print(f"\nEstatísticas do conjunto de treinamento normalizado:")
# Acessar os valores diretamente do array numpy retornado por all_values()
train_values = train_scaled.all_values(copy=False).flatten()
print(f"  - Média: {train_values.mean():.4f}")
print(f"  - Desvio padrão: {train_values.std():.4f}")
print(f"  - Mínimo: {train_values.min():.4f}")
print(f"  - Máximo: {train_values.max():.4f}")

# Nota: Covariáveis temporais (one-hot) já estão em escala 0-1 e não precisam de normalização

## 6. Configuração do Modelo TCN

Parâmetros chave:
- **input_chunk_length**: Quantas horas o modelo vê como entrada
- **output_chunk_length**: Quantas horas o modelo prevê de uma vez
- **kernel_size**: Tamanho do kernel convolucional
- **num_layers**: Número de camadas (controla o campo receptivo)
- **num_filters**: Número de filtros (capacidade do modelo)
- **dilation_base**: Base para dilatação exponencial (tipicamente 2)

**Estratégia de Previsão**:
- Se `output_chunk_length >= horizonte de previsão`: Previsão "one-shot" (rápida e estável)
- Se `output_chunk_length < horizonte de previsão`: Previsão autoregressiva (lenta, propaga erros)

In [ ]:
# Parâmetros do modelo
INPUT_CHUNK = 168   # 1 semana (7 dias * 24 horas)
OUTPUT_CHUNK = 24   # 1 dia (previsão de 24 horas)
KERNEL_SIZE = 3
NUM_LAYERS = 4      # RF = 31 horas com estes parâmetros
NUM_FILTERS = 64    # Aumentado do padrão de 3
DILATION_BASE = 2
DROPOUT = 0.2

# Imprimir configuração
print_model_config(
    input_chunk=INPUT_CHUNK,
    output_chunk=OUTPUT_CHUNK,
    kernel=KERNEL_SIZE,
    layers=NUM_LAYERS,
    filters=NUM_FILTERS,
    dilation=DILATION_BASE
)

# Configurar Early Stopping
early_stopper = EarlyStopping(
    monitor="val_loss",
    patience=10,
    min_delta=0.001,
    mode='min'
)

print("\n✓ Early stopping configurado (patience=10)")

## 7. Treinamento do Modelo

O modelo será treinado com:
- Dados de treinamento normalizados
- Covariáveis temporais
- Validação para monitoramento
- Early stopping para evitar overfitting

In [ ]:
# Criar modelo TCN
model = TCNModel(
    input_chunk_length=INPUT_CHUNK,
    output_chunk_length=OUTPUT_CHUNK,
    kernel_size=KERNEL_SIZE,
    num_filters=NUM_FILTERS,
    num_layers=NUM_LAYERS,
    dilation_base=DILATION_BASE,
    weight_norm=True,  # Estabiliza o treinamento
    dropout=DROPOUT,
    n_epochs=100,
    batch_size=32,
    optimizer_kwargs={'lr': 1e-3},
    pl_trainer_kwargs={
        'callbacks': [early_stopper],
        'accelerator': 'auto',
    },
    random_state=42,
    save_checkpoints=True,
    force_reset=True
)

print("✓ Modelo TCN criado")
print("\nIniciando treinamento...")

# Treinar modelo
model.fit(
    series=train_scaled,
    past_covariates=covariates,
    val_series=val_scaled,
    val_past_covariates=covariates,
    verbose=True
)

print("\n✓ Treinamento concluído!")

## 8. Avaliação no Conjunto de Validação

Vamos avaliar o desempenho do modelo no conjunto de validação.

In [ ]:
# Fazer previsões no conjunto de validação
# historical_forecasts precisa de contexto histórico antes do primeiro ponto de previsão.
# Como train e val são objetos separados, concatenamos para dar acesso ao histórico.
train_val_scaled = train_scaled.append(val_scaled)

# Fazer previsões começando no início do período de validação
# O parâmetro 'start' indica onde começar as previsões (após o treino)
val_predictions = model.historical_forecasts(
    series=train_val_scaled,
    past_covariates=covariates,
    forecast_horizon=OUTPUT_CHUNK,
    start=len(train_scaled),  # Índice onde a validação começa
    stride=OUTPUT_CHUNK,
    retrain=False,
    verbose=False
)

# Inverter normalização para escala original
val_predictions_original = scaler.inverse_transform(val_predictions)
val_original = scaler.inverse_transform(val_scaled)

# Calcular métricas
val_mae = mae(val_original, val_predictions_original)
val_rmse = rmse(val_original, val_predictions_original)
val_mape = mape(val_original, val_predictions_original)

print("Métricas no Conjunto de Validação:")
print(f"  - MAE:  {val_mae:.2f} MWmed")
print(f"  - RMSE: {val_rmse:.2f} MWmed")
print(f"  - MAPE: {val_mape:.2f}%")

# Plotar previsões vs real
plt.figure(figsize=(15, 6))
val_original.plot(label='Real', linewidth=2)
val_predictions_original.plot(label='Previsto', linewidth=2, linestyle='--')
plt.title('Validação: Real vs Previsto', fontsize=14)
plt.ylabel('Intercâmbio (MWmed)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Avaliação no Conjunto de Teste

Avaliação final no conjunto de teste (dados nunca vistos durante o treinamento).

In [ ]:
# Fazer previsões no conjunto de teste
# Concatenar train+val+test para fornecer contexto histórico completo
full_scaled = train_scaled.append(val_scaled).append(test_scaled)

# Fazer previsões começando no início do período de teste
test_predictions = model.historical_forecasts(
    series=full_scaled,
    past_covariates=covariates,
    forecast_horizon=OUTPUT_CHUNK,
    start=len(train_scaled) + len(val_scaled),  # Índice onde o teste começa
    stride=OUTPUT_CHUNK,
    retrain=False,
    verbose=False
)

# Inverter normalização para escala original
test_predictions_original = scaler.inverse_transform(test_predictions)
test_original = scaler.inverse_transform(test_scaled)

# Calcular métricas
test_mae = mae(test_original, test_predictions_original)
test_rmse = rmse(test_original, test_predictions_original)
test_mape = mape(test_original, test_predictions_original)

print("Métricas no Conjunto de Teste:")
print(f"  - MAE:  {test_mae:.2f} MWmed")
print(f"  - RMSE: {test_rmse:.2f} MWmed")
print(f"  - MAPE: {test_mape:.2f}%")

# Plotar previsões vs real
plt.figure(figsize=(15, 6))
test_original.plot(label='Real', linewidth=2)
test_predictions_original.plot(label='Previsto', linewidth=2, linestyle='--')
plt.title('Teste: Real vs Previsto', fontsize=14)
plt.ylabel('Intercâmbio (MWmed)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Previsão Futura

Vamos fazer uma previsão para as próximas 24 horas (1 dia) após o final dos dados.

In [ ]:
# Horizonte de previsão futuro
FORECAST_HORIZON = 24  # 24 horas (1 dia)

# Fazer previsão futura
# Usamos toda a série (normalizada) como contexto
full_series_scaled = scaler.transform(ts)

future_forecast = model.predict(
    n=FORECAST_HORIZON,
    series=full_series_scaled,
    past_covariates=covariates
)

# Inverter normalização
future_forecast_original = scaler.inverse_transform(future_forecast)

print(f"Previsão para as próximas {FORECAST_HORIZON} horas:")
print(f"Período: {future_forecast_original.start_time()} até {future_forecast_original.end_time()}")
print(f"\nEstatísticas da previsão:")
forecast_values = future_forecast_original.all_values(copy=False).flatten()
print(f"  - Média: {forecast_values.mean():.2f} MWmed")
print(f"  - Mínimo: {forecast_values.min():.2f} MWmed")
print(f"  - Máximo: {forecast_values.max():.2f} MWmed")

# Plotar últimas semanas + previsão
plt.figure(figsize=(15, 6))
ts[-7*24:].plot(label='Histórico (última semana)', linewidth=2)  # Última semana
future_forecast_original.plot(label=f'Previsão ({FORECAST_HORIZON}h)', linewidth=2, linestyle='--', marker='o')
plt.title(f'Previsão Futura - {ORIGIN} → {DESTINATION}', fontsize=14)
plt.ylabel('Intercâmbio (MWmed)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Análise Detalhada dos Resultados

In [ ]:
# Resumo das métricas
print("=" * 60)
print("RESUMO DAS MÉTRICAS")
print("=" * 60)
print(f"\n{'Conjunto':<15} {'MAE':<12} {'RMSE':<12} {'MAPE':<12}")
print("-" * 60)
print(f"{'Validação':<15} {val_mae:<12.2f} {val_rmse:<12.2f} {val_mape:<12.2f}")
print(f"{'Teste':<15} {test_mae:<12.2f} {test_rmse:<12.2f} {test_mape:<12.2f}")
print("=" * 60)

# Análise de resíduos no conjunto de teste
residuals = (test_original - test_predictions_original).all_values(copy=False).flatten()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de resíduos
axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero')
axes[0].set_xlabel('Resíduo (MWmed)', fontsize=12)
axes[0].set_ylabel('Frequência', fontsize=12)
axes[0].set_title('Distribuição dos Resíduos', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Q-Q plot (aproximado)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot (Normalidade dos Resíduos)', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nEstatísticas dos resíduos:")
print(f"  - Média: {residuals.mean():.2f} MWmed")
print(f"  - Desvio padrão: {residuals.std():.2f} MWmed")
print(f"  - Mínimo: {residuals.min():.2f} MWmed")
print(f"  - Máximo: {residuals.max():.2f} MWmed")

## 12. Conclusões e Considerações

### Pontos Fortes da Implementação:

1. **Arquitetura TCN**: Usa convoluções causais e dilatadas para capturar dependências temporais
2. **Campo Receptivo**: Calculado e validado para garantir que o modelo tenha memória suficiente
3. **Prevenção de Vazamento**: Normalização e divisão respeitam a ordem temporal
4. **Covariáveis Temporais**: Ajudam a capturar padrões sazonais (hora, dia, mês)
5. **One-Shot Forecasting**: `output_chunk_length = 24` permite previsões rápidas e estáveis

### Limitações:

1. **Apenas past_covariates**: TCN do Darts não suporta `future_covariates` ou `static_covariates`
2. **Horizonte Limitado**: Previsões muito longas podem perder precisão
3. **Dados Específicos**: Modelo treinado para um par específico de subsistemas

### Próximos Passos:

1. **Otimização de Hiperparâmetros**: Testar diferentes configurações de `num_layers`, `num_filters`, `kernel_size`
2. **Múltiplos Horizontes**: Treinar modelos para diferentes horizontes de previsão
3. **Ensemble**: Combinar previsões de múltiplos modelos
4. **Modelos Alternativos**: Comparar com RNN, Transformer, N-BEATS do Darts
5. **Análise Multi-Série**: Treinar um modelo global para múltiplos pares de subsistemas